# CS 5326 PA3 — Search over Reasoning: full benchmark pipeline

Runs all six configurations on the 50-problem GSM8K benchmark and writes the required artifacts:

| File | Content |
|---|---|
| `outputs/per_problem.jsonl` | one line per problem: greedy output, all 15 candidates, step scores / rationales / raw judge outputs for the 5 shared candidates, every method's answer |
| `outputs/results_summary.json` | 6-row method table (accuracy, calls, tokens, time) |
| `outputs/aggregation_ablation.json` | min / product / step_penalty at λ ∈ {0, 0.5, 2}, with tie rates |
| `outputs/sc_scaling_curve.json` | self-consistency accuracy for N = 1 … 15 |

**Models:** generator `Qwen/Qwen2.5-1.5B-Instruct` (fp16, `cuda:0`), judge `Qwen/Qwen2.5-7B-Instruct` (4-bit NF4, `cuda:1`).
**Seeds:** `transformers.set_seed(5326 + problem_index)` before each problem (0-based index).
**Resumable:** every finished problem is appended to `outputs/per_problem.jsonl` immediately; re-running skips finished problems.

All algorithmic code lives in `src/` (prompting, parsing, judge, aggregation, search, `src/pipeline.py` for the benchmark loop). This notebook only configures and calls it.

## 1. Configuration

In [ ]:
PROBLEM_LIMIT = None        # None = all 50 problems. Set e.g. 3 for a quick dry run.
JUDGE_BATCH_SIZE = 16       # judge prompts per GPU batch (auto-halves on CUDA OOM)
COMPLIANCE_GATE = 80.0      # stop if < this % of greedy outputs have a parseable Final Answer (None = never stop)
FRESH_START = False         # True = delete previous checkpoints/outputs and start over
RESTORE_FROM_INPUT = True   # Kaggle: resume from checkpoints found in an attached previous notebook output
CODE_DATASET_HINT = "cs5326-pa3-code"   # name of the Kaggle dataset holding src/, data/, tests/, submission_adapter.py

import time
RUN_START = time.time()

## 2. Locate the code, set the working directory, restore checkpoints

In [ ]:
import glob, json, os, shutil, sys

IN_KAGGLE = os.path.isdir("/kaggle/input")
CODE_ITEMS = ["src", "tests", "data", "submission_adapter.py"]

def find_code_root():
    # Running from the repository root (local Jupyter, or a git clone on Kaggle)
    if os.path.isfile("submission_adapter.py") and os.path.isdir("src"):
        return os.getcwd()
    hits = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
    hits = [h for h in hits if "/outputs/" not in h]
    preferred = [h for h in hits if CODE_DATASET_HINT in h]
    hits = sorted(preferred or hits, key=len)
    if not hits:
        raise FileNotFoundError(
            "PA3 code not found. On Kaggle: right sidebar -> Add Input -> select your "
            f"'{CODE_DATASET_HINT}' dataset."
        )
    return os.path.dirname(hits[0])

code_root = find_code_root()
if code_root.startswith("/kaggle/input"):
    # /kaggle/input is read-only: copy the code to /kaggle/working/pa3 (outputs/ is kept between runs)
    work = "/kaggle/working/pa3"
    os.makedirs(work, exist_ok=True)
    for item in CODE_ITEMS:
        src_p, dst_p = os.path.join(code_root, item), os.path.join(work, item)
        if os.path.isdir(dst_p):
            shutil.rmtree(dst_p)
        elif os.path.isfile(dst_p):
            os.remove(dst_p)
        (shutil.copytree if os.path.isdir(src_p) else shutil.copy2)(src_p, dst_p)
    os.chdir(work)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Code from:        ", code_root)
print("Working directory:", os.getcwd())

os.makedirs("outputs", exist_ok=True)
CHECKPOINTS = ["per_problem.jsonl", "greedy_checkpoint.jsonl"]
if FRESH_START:
    for name in CHECKPOINTS + ["results_summary.json", "aggregation_ablation.json", "sc_scaling_curve.json"]:
        p = os.path.join("outputs", name)
        if os.path.exists(p):
            os.remove(p)
            print("Deleted", p)
elif RESTORE_FROM_INPUT and IN_KAGGLE:
    for name in CHECKPOINTS:
        local = os.path.join("outputs", name)
        prev = sorted(glob.glob(f"/kaggle/input/**/outputs/{name}", recursive=True))
        if not os.path.exists(local) and prev:
            shutil.copy2(prev[0], local)
            print(f"Restored checkpoint {name} from {prev[0]}")
for name in CHECKPOINTS:
    p = os.path.join("outputs", name)
    n = sum(1 for _ in open(p, encoding="utf-8")) if os.path.exists(p) else 0
    print(f"{p}: {n} problems already done")

## 3. Dependencies (Kaggle already has torch / transformers / accelerate; only bitsandbytes must be upgraded)

In [ ]:
import subprocess
if IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes"], check=True)

## 4. Public smoke tests (offline, mock models) — confirms the right code version is loaded

In [ ]:
res = subprocess.run([sys.executable, "tests/test_public_adapter_smoke.py"], capture_output=True, text=True)
print(res.stdout[-1500:], res.stderr[-1500:])
assert res.returncode == 0, "Public smoke tests failed: wrong/old code version attached?"

## 5. Environment info (for report Section 5)

In [ ]:
import platform
import torch, transformers, accelerate
try:
    import bitsandbytes
    bnb_version = bitsandbytes.__version__
except Exception as e:
    bnb_version = f"import failed: {e}"

assert torch.cuda.is_available(), "No GPU. Kaggle: Settings -> Accelerator -> GPU T4 x2"
ENV = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bnb_version,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "gpu_memory_gb": [round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)
                      for i in range(torch.cuda.device_count())],
    "seed_protocol": "transformers.set_seed(5326 + problem_index), problem_index 0-based",
}
print(json.dumps(ENV, indent=2))

## 6. Load both models

In [ ]:
from src.utils import load_models, make_hf_generate_fn, make_hf_batch_judge_fn
from src.pipeline import CachedJudge

t0 = time.time()
gen_model, gen_tokenizer, judge_model, judge_tokenizer = load_models(generator_device="cuda:0", judge_device="cuda:1")
gen_model.eval(); judge_model.eval()
print(f"Models loaded in {time.time() - t0:.0f}s")
print("Generator device:", gen_model.device, "| Judge device:", judge_model.device)
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated {torch.cuda.memory_allocated(i) / 1e9:.2f} GB")
ENV["vram_after_load_gb"] = {f"cuda:{i}": round(torch.cuda.memory_allocated(i) / 1e9, 2) for i in range(torch.cuda.device_count())}

generate_fn = make_hf_generate_fn(gen_model, gen_tokenizer)
judge = CachedJudge(batch_judge_fn=make_hf_batch_judge_fn(judge_model, judge_tokenizer), batch_size=JUDGE_BATCH_SIZE)

## 7. Phase A — greedy on every problem + format-compliance check

The manual (§4.2) warns that Qwen2.5-1.5B can drift into markdown / LaTeX / `\boxed{}` and drop the `Final Answer:` line. Check the numbers below **before** the long run.

In [ ]:
from tqdm.auto import tqdm
from src.pipeline import load_benchmark, run_greedy_phase, compliance_report

rows = load_benchmark("data/gsm8k_test_50.jsonl")
assert len(rows) == 50, len(rows)
if PROBLEM_LIMIT:
    rows = rows[:PROBLEM_LIMIT]
print(f"Running on {len(rows)} problems")

t0 = time.time()
greedy_records = run_greedy_phase(rows, generate_fn, "outputs/greedy_checkpoint.jsonl", progress=tqdm)
print(f"Greedy phase: {time.time() - t0:.0f}s")

COMPLIANCE = compliance_report(greedy_records, rows)
print(json.dumps(COMPLIANCE, indent=2))

first = greedy_records[rows[0]["id"]]["greedy"]
print("\n--- Example greedy completion (problem 1) ---\n" + first["completion"])
print("parsed steps:", len(first["steps"]), "| answer:", first["answer"], "| gt:", rows[0]["ground_truth_answer"])
for pid in COMPLIANCE.get("unparsed_ids", [])[:3]:
    print(f"\n--- UNPARSED: {pid} ---\n" + greedy_records[pid]["greedy"]["completion"][-600:])

if COMPLIANCE_GATE is not None and COMPLIANCE["answer_parsed_rate_%"] < COMPLIANCE_GATE:
    raise RuntimeError(
        f"Only {COMPLIANCE['answer_parsed_rate_%']}% of greedy outputs have a parseable Final Answer "
        f"(gate {COMPLIANCE_GATE}%). Fix the prompt in src/generation.py before the long run."
    )

## 8. Phase B — SC@5, PRM judging, PRM Best-of-5 × 3 aggregations, ablation, SC@15

Per problem: 5 shared samples → judge every step once (batched + cached) → PRM rows reuse the same verdicts → 10 extra samples for SC@15. Each finished problem is checkpointed immediately.

In [ ]:
from src.pipeline import run_main_phase

t0 = time.time()
records = run_main_phase(rows, generate_fn, judge, greedy_records, "outputs/per_problem.jsonl", progress=tqdm)
MAIN_PHASE_SECONDS = time.time() - t0
print(f"Main phase: {MAIN_PHASE_SECONDS / 60:.1f} min for {len(records)} problems")

## 9. Build the artifacts

In [ ]:
from src.pipeline import read_jsonl, build_artifacts, format_table

wanted = {r["id"] for r in rows}
records = [r for r in read_jsonl("outputs/per_problem.jsonl") if r["id"] in wanted]
ART = build_artifacts(records, "outputs")

print(f"Problems evaluated: {ART['num_problems']}")
print("\nresults_summary.json\n" + format_table(ART["summary"]))
print("\naggregation_ablation.json\n" + format_table(ART["ablation"]))
print("\nsc_scaling_curve.json\n" + format_table(ART["curve"]))
print("\nJudge outputs without a 'Verdict:' line:", ART["judge_verdict_line_missing"])

## 10. Spot-check one problem and save run metadata

In [ ]:
rec = records[0]
print(f"{rec['id']}  ground truth = {rec['ground_truth_answer']}")
for c in rec["candidates"][:5]:
    print(f"\n[candidate {c['index']}] answer={c['answer']} step_scores={c['step_scores']}")
    for step, out in zip(c["steps"], c["judge_outputs"]):
        print(f"   STEP: {step[:110]}\n   JUDGE: {out.strip()[:160]!r}")
print("\nMethod answers:", {m: v["answer"] for m, v in rec["methods"].items()})

ENV["compliance"] = COMPLIANCE
ENV["judge_verdict_line_missing"] = ART["judge_verdict_line_missing"]
ENV["judge_batch_size"] = JUDGE_BATCH_SIZE
ENV["num_problems"] = ART["num_problems"]
ENV["notebook_wall_clock_minutes_this_session"] = round((time.time() - RUN_START) / 60, 1)
with open("outputs/run_environment.json", "w", encoding="utf-8") as f:
    json.dump(ENV, f, indent=2)
print("\nSaved outputs/run_environment.json")
print(sorted(os.listdir("outputs")))

## Done

Download from the notebook's **Output** tab (or `/kaggle/working/pa3/outputs/`):
`per_problem.jsonl`, `results_summary.json`, `aggregation_ablation.json`, `sc_scaling_curve.json` (+ `run_environment.json`, `greedy_checkpoint.jsonl` for your own analysis).